# Photodiode Transimpedance Amplifier (TIA) Analysis
 
 In this analysis the TIA is set up as follows:
 
 - **Feedback Network:**  
   - Resistor: **R_f = 2.5 MΩ**
   - Capacitor: **C_f = 10 nF**
 
   The transfer function (from photocurrent \( I_{pd}(s) \) to the op amp output \( V_{out}(s) \) ignoring DC bias)
   is given by:
 
   \[
   H(s) = \frac{V_{out}(s) - V_{\text{ref}}}{I_{pd}(s)} = \frac{R_f}{1 + s R_f C_f}
   \]
 
 - **Photodiode (SFH 213 FA) Parameters (from datasheet):**
   - Responsivity, \(S \approx 0.65\,\text{A/W}\)
   - Terminal capacitance: ~11 pF (small compared to C_f and will be neglected in low-frequency analysis)
 
 - **DC Bias:**  
   The photodiode is reverse-biased using a resistor divider (or a buffered 3.3 V reference) so that the noninverting input is at ~3.3 V.
 
 The following script sets the inputs, computes the pole frequency, expresses the transfer function,
 and plots the magnitude/phase response.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# For transfer function representation and Bode plot
try:
    from control import tf, bode_plot
except ImportError:
    print("Please install the 'control' library (e.g., pip install control) to run the following transfer function analysis.")


ModuleNotFoundError: No module named 'numpy'

In [ ]:
# ### 1. Define Input Variables

# Feedback network parameters
R_f = 1.5e6        # Feedback resistor in ohms (2.5 MΩ)
C_f = 10e-9        # Feedback capacitor in farads (10 nF)

# Photodiode parameters (from datasheet SFH 213 FA)
responsivity = 0.65  # in A/W (typical responsivity at 870 nm)

# DC bias/reference voltage at non-inverting input
V_ref = 3.3       # Volts

# Typical photocurrent under test conditions (e.g., 1 mW/cm^2 illumination)
I_ph_typ = 80e-6  # 80 microamps

In [ ]:
# ### 2. Compute Key Values
# 
# **Pole Frequency Calculation:**
# 
# The pole frequency \( f_p \) is:
# 
# \[
# f_p = \frac{1}{2\pi R_f C_f}
# \]

pole_freq = 1/(2*np.pi*R_f*C_f)
print("Pole frequency (Hz): {:.2f}".format(pole_freq))

In [ ]:
# ### 3. Define and Analyze the Transfer Function
# 
# The TIA transfer function is:
# 
# \[
# H(s) = \frac{R_f}{1 + s R_f C_f}
# \]
# 
# We’ll express this transfer function using the `control` library and also show a symbolic expression using sympy.

# Define the transfer function using the control library
try:
    num = [R_f]  # Numerator coefficient (DC gain)
    den = [R_f*C_f, 1]  # Denominator coefficients: s*R_f*C_f + 1
    H_tf = tf(num, den)
    print("Transfer Function H(s) =")
    print(H_tf)
except NameError:
    print("Control library not available. Skipping transfer function display.")


In [ ]:
# **Symbolic Transfer Function:**

# Create a symbolic variable s and derive H(s)
s = sp.symbols('s')
H_sym = R_f / (1 + R_f * C_f * s)
H_sym_simpl = sp.simplify(H_sym)
print("Symbolic Transfer Function H(s) =")
sp.pretty_print(H_sym_simpl)

In [ ]:
# ### 4. Bode Plot of the Transfer Function
# 
# Let’s plot the Bode magnitude and phase response over a frequency range.

if 'H_tf' in globals():
    # Generate Bode plot using the control library
    mag, phase, omega = bode_plot(H_tf, dB=True, Plot=False)  # Get data without plotting immediately

    # Convert omega (rad/s) to frequency in Hz for plotting
    frequencies = omega / (2 * np.pi)
    
    # Plot magnitude (in dB)
    plt.figure(figsize=(8, 4))
    plt.semilogx(frequencies, 20*np.log10(mag))
    plt.title("Bode Magnitude Plot")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Magnitude (dB)")
    plt.grid(True, which='both', ls='--')
    plt.show()
    
    # Plot phase (in degrees)
    plt.figure(figsize=(8, 4))
    plt.semilogx(frequencies, np.degrees(phase))
    plt.title("Bode Phase Plot")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Phase (degrees)")
    plt.grid(True, which='both', ls='--')
    plt.show()

In [ ]:
# ### 5. DC Output Voltage Calculation
# 
# For a given photocurrent \( I_{ph} \), the DC output voltage (ignoring the DC bias) is:
# 
# \[
# V_{out,dc} = I_{ph} \times R_f
# \]
# 
# Adding the DC bias, the output voltage will be:
# 
# \[
# V_{out} = V_{ref} + I_{ph} R_f
# \]

V_out_dc = V_ref + I_ph_typ * R_f
print("DC Output voltage due to typical photocurrent (V): {:.2f}".format(V_out_dc))



In [ ]:

# ### 6. Relationship between Incident Optical Power and Photocurrent
# 
# Using the photodiode responsivity \( S \) (A/W):
# 
# \[
# I_{ph} = S \times P_{in}
# \]
# 
# where \( P_{in} \) is the incident optical power (in watts).

def photocurrent(P_in_watts, responsivity=responsivity):
    """
    Calculate photocurrent (in amps) for a given incident optical power (in watts).
    """
    return responsivity * P_in_watts

# Example: Calculate photocurrent for 1 µW of incident light.
P_in_example = 1e-6  # 1 µW
I_ph_example = photocurrent(P_in_example)
print("Photocurrent for 1 µW input (A): {:.2e}".format(I_ph_example))

# Calculate expected output voltage change due to this photocurrent:
V_out_example = V_ref + I_ph_example * R_f
print("Expected output voltage for 1 µW input (V): {:.2f}".format(V_out_example))